# 07 — Compare rollout results across export keys

Loads every `rollout_results.json` under `Magic_window/Test&Rollout/*/`, builds summary tables, and saves comparison plots to `Test&Rollout/_comparison/`.

Re-run notebook 05 for each `EXPORT_KEY` first if a run is missing.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from compare_rollout_results import (
    comparison_output_dir,
    discover_rollout_results,
    load_rollout_results,
    metrics_to_long_df,
    run_comparison,
    summary_table,
)

MW_DIR = Path.cwd().resolve()
if MW_DIR.name != "Magic_window":
    MW_DIR = MW_DIR / "Magic_window"

paths = discover_rollout_results(MW_DIR)
print(f"Found {len(paths)} rollout_results.json files")
for p in paths:
    print(f"  {p.parent.name}")

In [ ]:
info = run_comparison(MW_DIR)
OUT_DIR = Path(info["output_dir"])
print(f"Wrote comparison artifacts to {OUT_DIR}")

## Summary table (full-field speed RMSE & skill)

In [ ]:
long_df = metrics_to_long_df(load_rollout_results(paths))
summary_df = summary_table(long_df)

display_cols = [
    "export_key",
    "group",
    "best_val_loss",
    "model_rmse_full_15d",
    "skill_score_full_15d",
    "model_rmse_full_30d",
    "skill_score_full_30d",
    "model_rmse_full_45d",
    "skill_score_full_45d",
]
summary_df[display_cols]

In [ ]:
center_pivot = long_df[long_df["region"] == "center"].pivot_table(
    index="export_key",
    columns="horizon_label",
    values=["model_rmse", "skill_score"],
)
full_pivot = long_df[long_df["region"] == "full"].pivot_table(
    index="export_key",
    columns="horizon_label",
    values=["model_rmse", "persistence_rmse", "skill_score"],
)
print("Center 4×4 — model RMSE & skill")
display(center_pivot)
print("Full field — persistence / model RMSE / skill")
display(full_pivot)

## Comparison plots

In [ ]:
for name, path in info["plots"].items():
    p = Path(path)
    print(name, p.name)
    display(Image(filename=str(p)))